# 01 ? Vector Database Fundamentals

This notebook begins the vector database section. It assumes you have studied embeddings, vector similarity, exact search, and ANN search. We will focus on the database mental model: what is stored, how records are organized, and what a database adds around vector search. Later notebooks cover CRUD in depth, persistence and indexes, Qdrant, filtering, and production concerns.


## Learning goals

By the end, you should be able to:

1. Explain why a vector index alone is not the whole database.
2. Describe a vector record as an ID, vector, and optional metadata/payload.
3. Explain what a collection is and why its records need a compatible vector configuration.
4. Trace how a RAG chunk moves from source text to a searchable record and back.
5. Identify key questions to answer before creating a collection.


## 1. Start with the RAG retrieval path

A RAG application does more than find a nearby vector. It must connect a query to original content, apply constraints, and return context for the language model. A simplified flow is:

**source documents ? chunks ? embeddings ? stored records ? nearest-neighbor candidates ? filtering/ranking ? source text for the prompt**

The embedding model maps content into vectors. A search algorithm compares vectors. A vector database manages those vectors together with IDs and metadata, and may provide persistence, filtering, indexing, replication, and an API.

The database usually stores the embedding and associated data; it does not automatically decide how to chunk documents or guarantee that an embedding model is right for your task. Those are application and model choices.


## 2. What makes a vector database different?

A basic in-memory search example can hold an array of vectors and return nearest row numbers. A RAG application also needs to know which document and chunk produced a vector, retrieve its text and source, restrict search to a tenant or document type, handle updates and deletion, survive restarts, and keep search usable as data grows.

A vector database adds data management and search capabilities around vector records. Products vary: some are dedicated vector databases, some are general databases with vector search, and some libraries focus on indexing/search rather than the full database lifecycle. We will compare examples later.


## 3. A vector record: ID, vector, and payload

A common conceptual record has three parts:

- **ID** identifies the record so it can be retrieved, updated, or deleted.
- **Vector** is the numeric embedding searched by similarity.
- **Payload / metadata** contains associated fields such as text, document ID, page, tenant, or timestamp.

Names vary by system. Qdrant calls a stored item a *point* and associated fields *payload*. Other systems use row, entity, or metadata. Think about each part's role, not only the product vocabulary.


In [ ]:
# Database-agnostic example of a chunk record.
# The numbers are illustrative, not generated by an embedding model.
record = {
    "id": "handbook-7:chunk-2",
    "vector": [0.12, -0.08, 0.31, 0.44],
    "payload": {
        "text": "Employees can reset a forgotten password from the sign-in page.",
        "document_id": "handbook-7",
        "chunk_index": 2,
        "section": "Account access",
        "tenant_id": "acme",
    },
}
record

The vector is what semantic similarity compares; the payload helps the application interpret and use the match. In a real collection, vector dimensions must match the embedding output, and the metric should match the model's intended use. The database cannot infer these choices from arbitrary numbers.

Consider whether full chunk text belongs in the payload or should be fetched from a separate document store using an ID. Both designs are used; the choice depends on retrieval, security, and operations needs.


## 4. Collections: groups of records with a search configuration

A **collection** (or equivalent product construct) groups records that share a search configuration. It commonly has a vector dimension, distance metric, and index/storage settings. It may also have a payload schema or indexes for frequently filtered fields.

A useful starting design is one collection for records created by the same embedding model and intended for the same retrieval use case. If you change embedding models or dimensions, do not assume old and new vectors can be searched together meaningfully. Equal dimensions do not guarantee compatible vector spaces.

A collection is a logical boundary, not necessarily the same thing as a tenant or a source document. Organization depends on isolation, scale, update patterns, and filtering needs.


## 5. Metadata is for meaning and constraints

Metadata is structured information about a record: `document_id`, `tenant_id`, `language`, `created_at`, `source`, or `chunk_index`. It can help show a source, filter candidates, update/delete all chunks from a document, and debug an unexpected result.

Metadata is not necessarily indexed just because it is stored. A database may need a field/payload index to make a filter efficient. Filters also affect correctness: a highly similar result from the wrong tenant is still a wrong result.


## 6. Where the database fits in a RAG system

1. Read source content and split it into chunks.
2. Attach stable IDs and useful metadata to each chunk.
3. Generate embeddings with a chosen model.
4. Write vectors and associated fields to a collection.
5. Embed a user query using a compatible query path.
6. Search top-K candidates, optionally applying filters.
7. Use IDs, scores, and payloads to build context for the LLM.
8. Update or delete records when the source changes.

IDs and metadata connect results back to source content and make maintenance possible. Later notebooks will implement the database-facing steps with Qdrant.


## 7. Keep these concepts separate

| Concept | Main job | Example question |
|---|---|---|
| Embedding model | Turns input into vectors | What vector represents this query or chunk? |
| Similarity metric | Scores vector relationships | Which vectors are closest under this metric? |
| Vector index | Organizes vectors for faster search | How can candidate vectors be found efficiently? |
| Vector database | Manages records and search/data operations | How do I store, find, filter, update, and persist records? |
| RAG application | Orchestrates retrieval and generation | Which evidence should go to the LLM, and how? |

A product may bundle several capabilities. The distinction helps locate problems: poor relevance may come from chunking, embeddings, metric choice, filters, index settings, or the query?not only the database.


## 8. Collection design checklist

Before creating a collection, write down:

1. **Record unit:** one chunk, image, or other retrieval unit?
2. **Stable ID:** how will changed source content map to records to update?
3. **Embedding model and dimension:** what happens if the model changes?
4. **Distance metric:** what does the model recommend?
5. **Payload fields:** which are needed for display, filtering, and maintenance?
6. **Filters:** which fields will be filtered often, and how many distinct values do they have?
7. **Original content location:** payload, document store, or both?
8. **Data boundaries:** users, tenants, teams, or document sets?
9. **Operational needs:** size, update frequency, persistence, backups, latency, and access control?


## 9. Knowledge check

Answer these before moving on:

1. If a search returns an ID and score but no text, what lookup is needed?
2. Why might two same-dimension embeddings still be unsuitable for one collection?
3. What is the difference between storing metadata and filtering on it efficiently?
4. Give an example where the nearest vector is still an invalid result.
5. Which parts of the RAG lifecycle belong to the model, database, and application?


### Suggested answers

1. Fetch the payload text or look up the ID in a document store.
2. Different models can encode meaning in incompatible vector spaces despite equal dimensions.
3. A stored field may need a database-specific index/configuration for efficient filtering.
4. A close match from another tenant, an outdated document, or the wrong language may be invalid.
5. The model produces embeddings; the database stores/manages/searches records; the application handles chunking, orchestration, business rules, and LLM context construction.


## Summary

A vector database manages searchable vector records together with the identifiers and structured fields needed to use and maintain them. The core concepts are **records**, **collections**, **vectors**, and **metadata/payloads**. Retrieval quality and correctness depend on choices across the whole RAG pipeline.

**Next:** `02_records_collections_metadata_crud.ipynb` will focus on record design and the create/read/update/delete lifecycle.
